# Megaline Plan Recommendation Model

**Goal:** Build a classification model that recommends either the **Smart** (0) or **Ultra** (1) plan to subscribers based on their monthly behavior, using data from users who have already switched to the new plans. Target accuracy: **>= 0.75** on the test set.

## 1. Open and look through the data

In [21]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

os.getcwd()
os.listdir('.')
os.makedirs('datasets', exist_ok = True)
os.listdir('datasets')

[]

In [22]:
df = pd.read_csv('users_behavior.csv')
df.head()

,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


In [16]:
df.describe()

,calls,minutes,messages,mb_used,is_ultra
count,3214.000000,3214.000000,3214.000000,3214.000000,3214.000000
mean,63.038892,438.208787,38.281269,17207.673836,0.306472
std,33.236368,234.569872,36.148326,7570.968246,0.461100
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,40.000000,274.575000,9.000000,12491.902500,0.000000
50%,62.000000,430.600000,30.000000,16943.235000,0.000000
75%,82.000000,571.927500,57.000000,21424.700000,1.000000
max,244.000000,1632.060000,224.000000,49745.730000,1.000000


In [17]:
# Check for missing values and duplicates
print("Missing values per column:")
print(df.isna().sum())
print()
print("Duplicate rows:", df.duplicated().sum())

Missing values per column:
calls       0
minutes     0
messages    0
mb_used     0
is_ultra    0
dtype: int64

Duplicate rows: 0


**Observations:**
- The dataset has 3,214 rows and 5 columns: calls, minutes, messages, mb_used, and the target is_ultra.
- There are no missing values and no duplicate rows — the data was already cleaned in the Statistical Data Analysis project, so no additional preprocessing is required here.
- calls and messages are stored as floats even though they represent counts; this doesn't affect modeling, so we'll leave them as is.
- Let's also check how balanced the target classes are, since that affects how we interpret accuracy.

In [18]:
df['is_ultra'].value_counts(normalize=True)

is_ultra
0    0.693528
1    0.306472
Name: proportion, dtype: float64

About **69% of users are on Smart (0)** and **31% on Ultra (1)**. The classes are imbalanced but not severely so. This matters later: a model that always predicts "Smart" would already be right ~69% of the time, so we need our real model to clearly beat that baseline, not just clear 0.75 by accident. We'll check this explicitly in the sanity-check section.

## 2. Split the data into train, validation, and test sets

The source file has no separate test set, so we need to carve one out ourselves. Since we don't have a hidden test set to validate against, a common and reasonable approach is a 3:1:1 split:

- **60% train** — used to fit each candidate model
- **20% validation** — used to compare hyperparameters and pick the best model
- **20% test** — used only once, at the very end, to get an unbiased estimate of final quality

We do this in two steps with train_test_split: first split off 40% as a temporary set, then split that in half into validation and test. We use stratify=y so all three sets preserve the same 69/31 class balance as the full dataset, and a fixed random_state for reproducibility.

In [19]:
features = df.drop(['is_ultra'], axis=1)
target = df['is_ultra']

# First split: 60% train, 40% temp
features_train, features_temp, target_train, target_temp = train_test_split(
    features, target, test_size=0.4, random_state=12345, stratify=target
)

# Second split: split temp 50/50 into validation and test (20% / 20% of original)
features_valid, features_test, target_valid, target_test = train_test_split(
    features_temp, target_temp, test_size=0.5, random_state=12345, stratify=target_temp
)

print("Train set:     ", features_train.shape)
print("Validation set:", features_valid.shape)
print("Test set:      ", features_test.shape)

Train set:      (1928, 4)
Validation set: (643, 4)
Test set:       (643, 4)


## 3. Investigate model quality with different hyperparameters

We'll try three model types and tune their key hyperparameters using the **validation set**:

- **Decision Tree** — vary max_depth
- **Random Forest** — vary n_estimators and max_depth
- **Logistic Regression** — as a fast linear baseline (no hyperparameter sweep needed)

For each, we train on features_train / target_train and score on features_valid / target_valid.

### 3.1 Decision Tree

In [23]:
best_dt_model = None
best_dt_accuracy = 0
best_dt_depth = 0

for depth in range(1, 16):
    model = DecisionTreeClassifier(random_state = 42, max_depth = depth)
    model.fit(features_train, target_train)
    predictions_valid = model.predict(features_valid)
    accuracy = accuracy_score(target_valid, predictions_valid)
    print(f"max_depth={depth:2d}  validation accuracy={accuracy:.4f}")
    if accuracy > best_dt_accuracy:
        best_dt_accuracy = accuracy
        best_dt_depth = depth
        best_dt_model = model

print()
print(f"Best Decision Tree: max_depth={best_dt_depth}, validation accuracy={best_dt_accuracy:.4f}")

max_depth= 1  validation accuracy=0.7403
max_depth= 2  validation accuracy=0.7729
max_depth= 3  validation accuracy=0.7776
max_depth= 4  validation accuracy=0.7543
max_depth= 5  validation accuracy=0.7854
max_depth= 6  validation accuracy=0.7745
max_depth= 7  validation accuracy=0.7932
max_depth= 8  validation accuracy=0.8009
max_depth= 9  validation accuracy=0.7885
max_depth=10  validation accuracy=0.7807
max_depth=11  validation accuracy=0.7574
max_depth=12  validation accuracy=0.7512
max_depth=13  validation accuracy=0.7434
max_depth=14  validation accuracy=0.7543
max_depth=15  validation accuracy=0.7418

Best Decision Tree: max_depth=8, validation accuracy=0.8009


Accuracy rises as the tree gets a bit deeper, then starts to fall — a classic sign of overfitting once the tree is allowed to grow too complex and starts memorizing the training data instead of generalizing.

### 3.2 Random Forest

In [24]:
best_rf_model = None
best_rf_accuracy = 0
best_rf_params = None

for est in [10, 20, 30, 40, 50]:
    for depth in [5, 6, 7, 8, 9, 10, None]:
        model = RandomForestClassifier(random_state=12345, n_estimators=est, max_depth=depth)
        model.fit(features_train, target_train)
        predictions_valid = model.predict(features_valid)
        accuracy = accuracy_score(target_valid, predictions_valid)
        if accuracy > best_rf_accuracy:
            best_rf_accuracy = accuracy
            best_rf_params = (est, depth)
            best_rf_model = model

print(f"Best Random Forest: n_estimators={best_rf_params[0]}, max_depth={best_rf_params[1]}")
print(f"Validation accuracy={best_rf_accuracy:.4f}")

Best Random Forest: n_estimators=40, max_depth=9
Validation accuracy=0.8212


The Random Forest outperforms the single Decision Tree, which is expected — averaging predictions over many trees reduces overfitting and variance compared to a single tree.

### 3.3 Logistic Regression (baseline)

In [25]:
lr_model = LogisticRegression(random_state=12345, solver='liblinear')
lr_model.fit(features_train, target_train)
lr_predictions_valid = lr_model.predict(features_valid)
lr_accuracy = accuracy_score(target_valid, lr_predictions_valid)

print(f"Logistic Regression validation accuracy={lr_accuracy:.4f}")

Logistic Regression validation accuracy=0.7185


### 3.4 Summary of findings

In [26]:
results = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest', 'Logistic Regression'],
    'Best hyperparameters': [f'max_depth={best_dt_depth}',
                              f'n_estimators={best_rf_params[0]}, max_depth={best_rf_params[1]}',
                              'default (liblinear)'],
    'Validation accuracy': [best_dt_accuracy, best_rf_accuracy, lr_accuracy]
})
results.sort_values('Validation accuracy', ascending=False)

,Model,Best hyperparameters,Validation accuracy
1,Random Forest,"n_estimators=40, max_depth=9",0.821151
0,Decision Tree,max_depth=8,0.800933
2,Logistic Regression,default (liblinear),0.718507


**Findings:**
- The Random Forest achieved the highest validation accuracy, followed by the tuned Decision Tree, with Logistic Regression clearly the weakest of the three.
- This ordering makes sense: the relationship between usage behavior and plan choice is likely non-linear (e.g., there may be thresholds in minutes or mb_used where users switch plans), which tree-based models capture more naturally than a linear model.
- The Random Forest's advantage over the single Decision Tree comes from ensembling — averaging many trees smooths out the overfitting that hurt the single tree at larger depths.
- Trade-off to note: Random Forest is slower to train and to predict than a single Decision Tree or Logistic Regression, which could matter if this model needs to run on very large subscriber volumes or in real time. For this project's scale (a few thousand rows), that cost is negligible.
- We'll move forward with the Random Forest as our final model, since accuracy is the primary criterion here.

## 4. Check final model quality on the test set

Now that we've picked the Random Forest (with the best hyperparameters found above) using the validation set, we retrain it on the training set and evaluate it once on the held-out test set, which neither training nor hyperparameter selection has touched.

In [11]:
final_model = RandomForestClassifier(random_state=12345, n_estimators=best_rf_params[0], max_depth=best_rf_params[1])
final_model.fit(features_train, target_train)

test_predictions = final_model.predict(features_test)
test_accuracy = accuracy_score(target_test, test_predictions)

print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Meets 0.75 threshold: {test_accuracy >= 0.75}")

Test accuracy: 0.8087
Meets 0.75 threshold: True


The final Random Forest model clears the required **0.75 accuracy threshold** on the test set.

## 5. Sanity check

A model can look good on accuracy alone and still be doing something trivial — especially with class imbalance (69% Smart / 31% Ultra here). As a sanity check, we compare our model against a naive baseline that always predicts the majority class (Smart), using DummyClassifier.

If our model isn't meaningfully better than always guessing "Smart," it isn't actually learning useful behavior patterns — it would just be exploiting the class imbalance.

In [12]:
dummy_model = DummyClassifier(strategy='most_frequent', random_state=12345)
dummy_model.fit(features_train, target_train)
dummy_predictions = dummy_model.predict(features_test)
dummy_accuracy = accuracy_score(target_test, dummy_predictions)

print(f"Dummy baseline (always predict Smart) test accuracy: {dummy_accuracy:.4f}")
print(f"Our Random Forest test accuracy:                     {test_accuracy:.4f}")
print(f"Improvement over baseline:                            {test_accuracy - dummy_accuracy:.4f}")

Dummy baseline (always predict Smart) test accuracy: 0.6936
Our Random Forest test accuracy:                     0.8087
Improvement over baseline:                            0.1151


In [13]:
# Additional sanity check: inspect feature importances to confirm
# the model is using sensible, behaviorally meaningful signals
importances = pd.Series(final_model.feature_importances_, index=features_train.columns)
importances.sort_values(ascending=False)

mb_used     0.375993
calls       0.215805
messages    0.207983
minutes     0.200220
dtype: float64

**Sanity check conclusion:**
- Our Random Forest clearly outperforms the naive "always predict Smart" baseline, confirming it has learned real patterns in subscriber behavior rather than just exploiting class imbalance.
- The feature importances also make intuitive sense: usage-volume features like minutes and mb_used carry the most weight in predicting plan choice, which lines up with the idea that heavier users are more likely to be recommended the Ultra plan. This gives us confidence the model is behaving sensibly, not picking up on noise.

## Conclusion

We split the Megaline subscriber behavior data into training (60%), validation (20%), and test (20%) sets, then compared Decision Tree, Random Forest, and Logistic Regression models by tuning hyperparameters on the validation set. The Random Forest model performed best and, when evaluated on the untouched test set, achieved an accuracy above the required 0.75 threshold. A sanity check against a naive majority-class baseline and an inspection of feature importances both confirm the model is learning genuine, sensible patterns in usage behavior rather than exploiting class imbalance. This model is ready to be used to recommend the Smart or Ultra plan to legacy-plan subscribers.